# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [14]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1- Read tic-tac-toe.csv into a dataframe.
df = pd.read_csv('tic-tac-toe.csv')

# 2- Inspect the dataset. 
# Checking first rows and data types to ensure reliability.
print("Dataset Head:")
print(df.head())
print("\nDataset Info:")
print(df.info())

# 3- Convert the categorical values to numeric in all columns.
# Mapping board marks: x = 1, o = -1, empty (b) = 0
mapping = {'x': 1, 'o': -1, 'b': 0}
board_columns = ['TL', 'TM', 'TR', 'ML', 'MM', 'MR', 'BL', 'BM', 'BR']

for col in board_columns:
    df[col] = df[col].map(mapping)

# Encoding the 'class' column (True -> 1, False -> 0)
le = LabelEncoder()
df['class'] = le.fit_transform(df['class'])

# 4- Separate the inputs and output.
X = df.drop('class', axis=1)  # Features (the 9 grid positions)
y = df['class']               # Target (Win/Loss-Draw)

# 5- Normalize the input data.
# Standardizing the features to have a mean of 0 and variance of 1.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nProcessed Input (First 5 rows):")
print(X_scaled[:5])

Dataset Head:
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB
None

Processed Input (First 5 rows):
[[ 1.03516957  1.10682993  1.03516957  1.10682993 -1.24199419 -1.2235944
   1.03516957 -1.2235944  -1.23155602]
 [ 1.03516957  1.1

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [15]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1- Split the training and test data.
# We'll use 20% of the data for testing (test_size=0.2)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2- Create a Sequential model.
model = tf.keras.models.Sequential()

# 3- Add several layers to your model. 
# We start with an input layer matching our 9 features.
model.add(tf.keras.layers.Dense(128, activation='relu', input_shape=(9,))) # Input + Hidden Layer 1
model.add(tf.keras.layers.Dense(64, activation='relu'))                  # Hidden Layer 2
model.add(tf.keras.layers.Dense(32, activation='relu'))                  # Hidden Layer 3

# Output layer: 2 units (Class 0 and Class 1) using Softmax
model.add(tf.keras.layers.Dense(2, activation='softmax'))

# 4- Compile the model.
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5- Fit the training data.
# 50 epochs is usually enough for this small dataset to reach high accuracy.
print("Starting training...")
model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

# 6- Evaluate your neural network model with the test data.
print("\nEvaluation on Test Data:")
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# 7- Save your model as tic-tac-toe.model.
model.save('tic-tac-toe.keras')
print("\nModel saved successfully!")

Starting training...
Epoch 1/50


c:\Users\pablo\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6697 - loss: 0.6065
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7728 - loss: 0.5025 
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8211 - loss: 0.4157 
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9034 - loss: 0.3095 
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9726 - loss: 0.1982 
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9883 - loss: 0.1037 
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9948 - loss: 0.0535 
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0264 
Epoch 9/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0152 
Epoch 10/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0100 
Epoch 11/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0069 
Epoch 12/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - los

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [16]:
import tensorflow as tf
import numpy as np

# 1- Load your saved model
model = tf.keras.models.load_model('tic-tac-toe.keras')

# 2- Pick a few random rows from the test dataset
# We will take 5 random samples from our X_test and y_test
random_indices = np.random.choice(len(X_test), size=5, replace=False)
samples = X_test[random_indices]
true_labels = y_test.iloc[random_indices].values

# 3- Make predictions
predictions = model.predict(samples)

# 4- Check if the predictions are correct
print("\n--- Prediction Results ---")
for i in range(len(samples)):
    # Get the index of the highest probability (0 or 1)
    predicted_class = np.argmax(predictions[i])
    
    # Map back to readable text
    result = "Player X Wins" if predicted_class == 1 else "Loss or Draw"
    actual = "Player X Wins" if true_labels[i] == 1 else "Loss or Draw"
    
    print(f"Sample {i+1}:")
    print(f"  Predicted: {result} (Confidence: {np.max(predictions[i])*100:.2f}%)")
    print(f"  Actual:    {actual}")
    print(f"  Correct?   {predicted_class == true_labels[i]}")
    print("-" * 25)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step

--- Prediction Results ---
Sample 1:
  Predicted: Loss or Draw (Confidence: 100.00%)
  Actual:    Loss or Draw
  Correct?   True
-------------------------
Sample 2:
  Predicted: Loss or Draw (Confidence: 100.00%)
  Actual:    Loss or Draw
  Correct?   True
-------------------------
Sample 3:
  Predicted: Player X Wins (Confidence: 99.97%)
  Actual:    Player X Wins
  Correct?   True
-------------------------
Sample 4:
  Predicted: Player X Wins (Confidence: 99.97%)
  Actual:    Player X Wins
  Correct?   True
-------------------------
Sample 5:
  Predicted: Player X Wins (Confidence: 100.00%)
  Actual:    Player X Wins
  Correct?   True
-------------------------


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [17]:
import tensorflow as tf

# 1- Create a new Sequential model with more layers
model_improved = tf.keras.models.Sequential([
    # Input layer + first hidden layer
    tf.keras.layers.Dense(256, activation='relu', input_shape=(9,)),
    # Adding more depth (extra layers) as requested
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    # Output layer
    tf.keras.layers.Dense(2, activation='softmax')
])

# 2- Adjust the learning rate
# We create a custom Adam instance. 
# 0.001 is default; 0.0005 is slower/more precise; 0.01 is faster/riskier.
custom_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

# 3- Compile with the custom optimizer
model_improved.compile(
    optimizer=custom_optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4- Adjust the number of epochs
# Training for 100 epochs to see if it reaches the "ceiling" earlier
print("Training improved model...")
history = model_improved.fit(
    X_train, 
    y_train, 
    epochs=100, 
    batch_size=32, 
    validation_data=(X_test, y_test), # Adding validation to monitor performance
    verbose=1
)

# 5- Evaluate
loss, accuracy = model_improved.evaluate(X_test, y_test)
print(f"\nImproved Model - Test Accuracy: {accuracy:.4f}, Test Loss: {loss:.4f}")

Training improved model...
Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6945 - loss: 0.6247 - val_accuracy: 0.7031 - val_loss: 0.5736
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7950 - loss: 0.4980 - val_accuracy: 0.7917 - val_loss: 0.4733
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8603 - loss: 0.3774 - val_accuracy: 0.8906 - val_loss: 0.3440
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9426 - loss: 0.2505 - val_accuracy: 0.9479 - val_loss: 0.2243
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9817 - loss: 0.1312 - val_accuracy: 0.9844 - val_loss: 0.1120
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9922 - loss: 0.0617 - val_accuracy: 0.9844 - val_loss: 0.0666
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9987 - loss: 0.0263 - val_accuracy: 0.9844 - val_loss: 0.0452
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.013

**Which approach(es) did you find helpful to improve your model performance?**

In [18]:
# The most helpful approach was adjusting the Learning Rate and increasing the number of Epochs. While the initial model was strong, 
# fine-tuning the learning rate allowed the optimizer to converge more stably, 
# and increasing the epochs ensured the model had enough iterations to achieve near-total confidence (100%) on the test set. 
# Adding layers provided the necessary complexity to capture all game-winning patterns without overfitting.